In [2]:
import pandas as pd

In [3]:
df = pd.read_csv('/Users/davidgheorghe/Documents/DSIT Research Paper/GITHUB/GITHUB API/API2/github_osint_results.csv')

In [4]:
df.shape

(71395, 8)

In [5]:
df.head()

,type,query,name,url,description,stars,language,date
0,repository,"""open source"" ""artificial intelligence"" ""secur...",mahseema/awesome-ai-tools,https://github.com/mahseema/awesome-ai-tools,A curated list of Artificial Intelligence Top ...,3923.0,NaN,2025-12-17T11:57:15Z
1,repository,"""open source"" ""artificial intelligence"" ""secur...",ottosulin/awesome-ai-security,https://github.com/ottosulin/awesome-ai-security,A collection of awesome resources related AI s...,374.0,NaN,2025-12-17T05:10:57Z
2,repository,"""open source"" ""artificial intelligence"" ""secur...",mikeroyal/Open-Source-Security-Guide,https://github.com/mikeroyal/Open-Source-Secur...,Open Source Security Guide. Learn all about Se...,1034.0,Go,2025-12-16T10:26:10Z
3,repository,"""open source"" ""artificial intelligence"" ""secur...",CoderSJX/AI-Resources-Central,https://github.com/CoderSJX/AI-Resources-Central,Bringing together outstanding artificial inte...,1127.0,NaN,2025-12-17T14:05:02Z
4,repository,"""open source"" ""artificial intelligence"" ""secur...",visenger/awesome-mlops,https://github.com/visenger/awesome-mlops,A curated list of references for MLOps,13475.0,NaN,2025-12-15T18:08:02Z


In [6]:
df.isna().sum()

type               0
query              0
name               0
url                0
description     5473
stars           2369
language       38022
date               0
dtype: int64

In [7]:
df2 = df.dropna()

In [8]:
df2.shape

(30815, 8)

In [9]:
df2['text_all'] = (
    df2['name'].fillna('') + ' ' +
    df2['description'].fillna('') + ' ' +
    df2['query'].fillna('')
).str.lower()

/var/folders/03/6nkjsrk54q3f89x4qy6tnl280000gn/T/ipykernel_89314/1091143480.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2['text_all'] = (


In [10]:
# AI-related keywords
ai_keywords = [
    "ai", "artificial intelligence",
    "machine learning", "ml",
    "deep learning",
    "neural network", "neural networks",
    "large language model", "llm",
    "generative ai",
    "foundation model"
]

# Security-related keywords
security_keywords = [
    "security", "secure",
    "vulnerability", "vulnerabilities", "vulnerable",
    "exploit", "attacker", "attackers", "attack", "attacks",
    "adversarial",           # often used in "adversarial attacks"
    "penetration testing", "pentest", "pen-test",
    "malware", "threat", "threats",
    "xss", "sql injection", "csrf",
    "injection",
    "privilege escalation",
    "red team",
    "defense", "defence",
    "hardening",
    "cybersecurity", "cyber security",
    "infosec",
    "security auditing", "security audit", "security tool",
    "cve", "cvss"
]

In [11]:
def make_pattern(keywords):
    parts = []
    for k in keywords:
        k_ = k.replace(" ", r"\s+")
        parts.append(r"\b" + k_ + r"\b")
    pattern = "(?:" + "|".join(parts) + ")"
    return pattern

In [12]:
ai_pattern = make_pattern(ai_keywords)
security_pattern = make_pattern(security_keywords)

In [13]:
df2['is_ai'] = df2['text_all'].str.contains(ai_pattern, regex=True, na=False)
df2['is_security'] = df2['text_all'].str.contains(security_pattern, regex=True, na=False)

df2['is_ai'].value_counts(), df2['is_security'].value_counts()


/var/folders/03/6nkjsrk54q3f89x4qy6tnl280000gn/T/ipykernel_89314/1154363465.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2['is_ai'] = df2['text_all'].str.contains(ai_pattern, regex=True, na=False)
/var/folders/03/6nkjsrk54q3f89x4qy6tnl280000gn/T/ipykernel_89314/1154363465.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2['is_security'] = df2['text_all'].str.contains(security_pattern, regex=True, na=False)


(is_ai
 True    30815
 Name: count, dtype: int64,
 is_security
 True     19903
 False    10912
 Name: count, dtype: int64)

In [14]:
df2['is_intersection'] = df2['is_ai'] & df2['is_security']
intersection_df = df2[df2['is_intersection']]

len(intersection_df), len(df2)


/var/folders/03/6nkjsrk54q3f89x4qy6tnl280000gn/T/ipykernel_89314/443220505.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2['is_intersection'] = df2['is_ai'] & df2['is_security']


(19903, 30815)

In [15]:
intersection_df.duplicated(subset='name').sum()

17831

In [16]:
intersection_df = df2[df2['is_intersection']].copy()


In [17]:
unique_df = intersection_df.drop_duplicates(subset='name', keep='first').reset_index(drop=True)


In [18]:
unique_df

,type,query,name,url,description,stars,language,date,text_all,is_ai,is_security,is_intersection
0,repository,"""open source"" ""artificial intelligence"" ""secur...",mikeroyal/Open-Source-Security-Guide,https://github.com/mikeroyal/Open-Source-Secur...,Open Source Security Guide. Learn all about Se...,1034.0,Go,2025-12-16T10:26:10Z,mikeroyal/open-source-security-guide open sour...,True,True,True
1,repository,"""open source"" ""artificial intelligence"" ""secur...",skills/secure-code-game,https://github.com/skills/secure-code-game,"A GitHub Security Lab initiative, providing an...",2554.0,JavaScript,2025-12-17T10:37:34Z,skills/secure-code-game a github security lab ...,True,True,True
2,repository,"""open source"" ""artificial intelligence"" ""secur...",BrainCog-X/Brain-Cog,https://github.com/BrainCog-X/Brain-Cog,Brain-inspired Cognitive Intelligence Engine (...,587.0,Python,2025-12-16T07:32:55Z,braincog-x/brain-cog brain-inspired cognitive ...,True,True,True
3,repository,"""open source"" ""artificial intelligence"" ""secur...",lfnovo/open-notebook,https://github.com/lfnovo/open-notebook,An Open Source implementation of Notebook LM w...,15304.0,TypeScript,2025-12-17T14:17:18Z,lfnovo/open-notebook an open source implementa...,True,True,True
4,repository,"""open source"" ""artificial intelligence"" ""secur...",OWASP/www-project-ai-testing-guide,https://github.com/OWASP/www-project-ai-testin...,OWASP Foundation web repository,596.0,Python,2025-12-16T04:48:17Z,owasp/www-project-ai-testing-guide owasp found...,True,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...
2067,repository,"""open source software"" ""machine learning"" ""sec...",aws-samples/aws-cdk-pipelines-datalake-etl,https://github.com/aws-samples/aws-cdk-pipelin...,This solution helps you deploy ETL jobs on dat...,68.0,Python,2025-08-28T14:50:36Z,aws-samples/aws-cdk-pipelines-datalake-etl thi...,True,True,True
2068,repository,"""open source software"" ""machine learning"" ""sec...",srmainwaring/curio,https://github.com/srmainwaring/curio,ROS packages to control a version of Roger Che...,63.0,Python,2025-11-10T04:32:19Z,srmainwaring/curio ros packages to control a v...,True,True,True
2069,repository,"""open source software"" ""LLM"" ""security"" stars:...",r3-team/r3,https://github.com/r3-team/r3,REI3 - Free and open low code,506.0,JavaScript,2025-12-16T00:48:28Z,"r3-team/r3 rei3 - free and open low code ""open...",True,True,True
2070,repository,"""open source software"" ""LLM"" ""security"" stars:...",i-am-bee/beeai-code-interpreter,https://github.com/i-am-bee/beeai-code-interpr...,An HTTP service intended as a backend for an L...,69.0,Python,2025-12-05T00:52:33Z,i-am-bee/beeai-code-interpreter an http servic...,True,True,True


In [21]:
unique_df['stars'] = unique_df['stars'].astype(int)

In [22]:
unique_df

,type,query,name,url,description,stars,language,date,text_all,is_ai,is_security,is_intersection
0,repository,"""open source"" ""artificial intelligence"" ""secur...",mikeroyal/Open-Source-Security-Guide,https://github.com/mikeroyal/Open-Source-Secur...,Open Source Security Guide. Learn all about Se...,1034,Go,2025-12-16T10:26:10Z,mikeroyal/open-source-security-guide open sour...,True,True,True
1,repository,"""open source"" ""artificial intelligence"" ""secur...",skills/secure-code-game,https://github.com/skills/secure-code-game,"A GitHub Security Lab initiative, providing an...",2554,JavaScript,2025-12-17T10:37:34Z,skills/secure-code-game a github security lab ...,True,True,True
2,repository,"""open source"" ""artificial intelligence"" ""secur...",BrainCog-X/Brain-Cog,https://github.com/BrainCog-X/Brain-Cog,Brain-inspired Cognitive Intelligence Engine (...,587,Python,2025-12-16T07:32:55Z,braincog-x/brain-cog brain-inspired cognitive ...,True,True,True
3,repository,"""open source"" ""artificial intelligence"" ""secur...",lfnovo/open-notebook,https://github.com/lfnovo/open-notebook,An Open Source implementation of Notebook LM w...,15304,TypeScript,2025-12-17T14:17:18Z,lfnovo/open-notebook an open source implementa...,True,True,True
4,repository,"""open source"" ""artificial intelligence"" ""secur...",OWASP/www-project-ai-testing-guide,https://github.com/OWASP/www-project-ai-testin...,OWASP Foundation web repository,596,Python,2025-12-16T04:48:17Z,owasp/www-project-ai-testing-guide owasp found...,True,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...
2067,repository,"""open source software"" ""machine learning"" ""sec...",aws-samples/aws-cdk-pipelines-datalake-etl,https://github.com/aws-samples/aws-cdk-pipelin...,This solution helps you deploy ETL jobs on dat...,68,Python,2025-08-28T14:50:36Z,aws-samples/aws-cdk-pipelines-datalake-etl thi...,True,True,True
2068,repository,"""open source software"" ""machine learning"" ""sec...",srmainwaring/curio,https://github.com/srmainwaring/curio,ROS packages to control a version of Roger Che...,63,Python,2025-11-10T04:32:19Z,srmainwaring/curio ros packages to control a v...,True,True,True
2069,repository,"""open source software"" ""LLM"" ""security"" stars:...",r3-team/r3,https://github.com/r3-team/r3,REI3 - Free and open low code,506,JavaScript,2025-12-16T00:48:28Z,"r3-team/r3 rei3 - free and open low code ""open...",True,True,True
2070,repository,"""open source software"" ""LLM"" ""security"" stars:...",i-am-bee/beeai-code-interpreter,https://github.com/i-am-bee/beeai-code-interpr...,An HTTP service intended as a backend for an L...,69,Python,2025-12-05T00:52:33Z,i-am-bee/beeai-code-interpreter an http servic...,True,True,True


In [23]:
unique_df.to_csv("/Users/davidgheorghe/Downloads/GitHub_Filtered_Dataset.csv", index=False)
